# Medical Diagnosis: Diabetes Prediction

**Objective:**
Predict the onset of diabetes from diagnostic measurements, with interpretability and calibration suitable for clinical decision support.

**Dataset**
UCI Pima Indians Diabetes Database:
* 768 samples, 8 clinical features
* Binary target: diabetic (1) or not (0)
* ~35% positive class
* Available: sklearn.datasets or direct download

### Importing Dependencies

In [32]:
pip install shap

   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/555.9 kB ? eta -:--:--
   ------------------ --------------------- 262.1/555.9 kB ? eta -:--:--
   ----------------------------------- -- 524.3/555.9 kB 636.0 kB/s eta 0:00:01
   ---------------------------------------- 555.9/555.9 kB 663.0 kB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.1 MB ? eta -:--:--
    --------------------------------------- 0.5/38.1 MB 1.1 MB/s eta 0:00:35
    --------------------------------------- 0.8/38.1 MB 989.6 kB/s eta 0:00:38
   - -------------------------------------- 1.3/38.1 MB 1.3 MB/s eta 0:00:30
   - -------------------------------------- 1.6/38.1 MB 1.3 MB/s eta 0:00:28
   - ------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import shap

import numpy as np
import pandas as pd
import json
import os

from sklearn.datasets import fetch_openml
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve

#### Setting Config

In [2]:
def set_seed(seed = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

#### Loading Data

In [3]:
def load_data():
    data = fetch_openml(name = 'diabetes', version = 1, as_frame = True, parser = 'auto')
    
    df = data.frame
    target = data.target_names[0]
    df[target] = (df[target] == 'tested_positive').astype(int)
    df.rename(columns = {target : 'Outcome'}, inplace = True)
    print(df.shape)

    return df


df = load_data()
df.sample(8)

(768, 9)


,preg,plas,pres,skin,insu,mass,pedi,age,Outcome
668,6,98,58,33,190,34.0,0.430,43,0
324,2,112,75,32,0,35.7,0.148,21,0
624,2,108,64,0,0,30.8,0.158,21,0
690,8,107,80,0,0,24.6,0.856,34,0
473,7,136,90,0,0,29.9,0.210,50,0
204,6,103,72,32,190,37.7,0.324,55,0
97,1,71,48,18,76,20.4,0.323,22,0
336,0,117,0,0,0,33.8,0.932,44,0


In [4]:
df.Outcome.unique()

array([1, 0])

In [5]:
df.columns

Index(['preg', 'plas', 'pres', 'skin', 'insu', 'mass', 'pedi', 'age',
       'Outcome'],
      dtype='object')

In [6]:
df.rename(columns = {'preg': 'Pregnancies', 'plas': 'Glucose', 'pres': 'Blood Pressure', 
                          'insu': 'Insulin', 'mass': 'BMI', 'pedi': 'DiabetesPedigreeFunction', 
                          'age': 'Age', 'skin': 'SkinThickness'}, inplace = True)

In [7]:
df.columns

Index(['Pregnancies', 'Glucose', 'Blood Pressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [8]:
df.describe()

,Pregnancies,Glucose,Blood Pressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   Blood Pressure            768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [10]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
Blood Pressure              0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [11]:
print(f'Diabetic: {(df.Outcome == 1).count()}')
print(f'Non Diabetic: {(df.Outcome == 0).count()}')

Diabetic: 768
Non Diabetic: 768


#### Preprocessing

In [12]:
X = df.iloc[:, 0: 8]
y = df.Outcome

# Stratify because right now i am using just sample data but original data is imbalanced
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.2, random_state = 42,
                                                 stratify = y_temp)

print(f'Train: {len(X_train)}, Test: {len(X_test)}, Val: {len(X_val)}')
print(f'Train Positive Rate: {y_train.mean():.2f}')

Train: 491, Test: 154, Val: 123
Train Positive Rate: 0.35


In [13]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

y_train = np.array(y_train)
y_val   = np.array(y_val)
y_test  = np.array(y_test)

#### Model

In [14]:
class DiabetesNet(nn.Module):
    def __init__(self, in_features:int = 8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(32, 1)
        )

        for i in self.modules():
            if isinstance(i, nn.Linear):
                nn.init.kaiming_normal_(i.weight, nonlinearity = 'relu')
                nn.init.zeros_(i.bias)

    def forward(self, x):
        return self.net(x)

In [15]:
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [16]:
train_ds = DiabetesDataset(X_train_s, y_train)
val_ds = DiabetesDataset(X_val_s, y_val)
test_ds = DiabetesDataset(X_test_s, y_test)

train_loader = DataLoader(train_ds, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 64, shuffle = False)
test_loader = DataLoader(test_ds, batch_size = 64, shuffle = False)

#### Training

In [17]:
model = DiabetesNet(in_features = 8).to(DEVICE)

pos_wt = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_wt)
optimizer = optim.AdamW(model.parameters(), lr = 0.0005, weight_decay = 0.01)

print(f'Total Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'pos_weight: {pos_wt.item():.3f}')

Total Parameters: 2,881
pos_weight: 1.871


In [21]:
def run_epoch_diabetes(model, loader, criterion, optimizer = None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for bX, by in loader:
            bX, by = bX.to(DEVICE), by.to(DEVICE)
            if is_training:
                optimizer.zero_grad()
            logits = model(bX)
            loss = criterion(logits, by)
            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * bX.size(0)
            all_probs.append(torch.sigmoid(logits).detach().cpu().squeeze())
            all_labels.append(by.cpu().squeeze())

        return (total_loss / len(loader.dataset),
                np.concatenate(all_probs),
                np.concatenate(all_labels).astype(int))

In [24]:
def clinical_metrics(y_true, y_prob, threshold = 0.5):
    eps = 1e-8
    
    pred = (y_prob >= threshold).astype(int)
    tp = ((pred == 1) & (y_true == 1)).sum()
    tn = ((pred == 0) & (y_true == 0)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    fn = ((pred == 0) & (y_true == 1)).sum()

    sensitivity = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    ppv = tp / (tp + fp + eps)
    npv = tn / (tn + fn + eps)
    acc = (tp + tn) / (tp + tn + fp + fn + eps)
    f1 = 2 * sensitivity * ppv / (sensitivity + ppv + eps)

    return {
        'sensitivity': sensitivity, 'specificity': specificity, 'ppv': ppv, 'npv': npv,
        'accuracy': acc, 'f1': f1, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn
    }

In [26]:
EPOCHS = 100
best_val_loss = float('inf')
patience = 15
patience_count = 0

os.makedirs('project2_diabetes', exist_ok = True)

for epoch in range(EPOCHS):
    train_loss, _, _ = run_epoch_diabetes(model, train_loader, criterion, optimizer)
    val_loss, v_probs, v_lbls = run_epoch_diabetes(model, val_loader, criterion)

    vm = clinical_metrics(v_lbls, v_probs, threshold = 0.4)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save(model.state_dict(), 'project2_diabetes/best_model.pth')
    else:
        patience_count += 1

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>4} | "
              f"Loss {train_loss:.4f}/{val_loss:.4f} | "
              f"Sens {vm['sensitivity']:.4f} | "
              f"Spec {vm['specificity']:.4f} | "
              f"F1 {vm['f1']:.4f}")

    if patience_count >= patience:
        print(f'Early Stopping at epoch {epoch + 1}')
        break

Epoch   20 | Loss 0.7738/0.6841 | Sens 0.9535 | Spec 0.5750 | F1 0.6949
Epoch   40 | Loss 0.7283/0.6327 | Sens 0.9767 | Spec 0.6125 | F1 0.7241
Epoch   60 | Loss 0.7171/0.6059 | Sens 0.9070 | Spec 0.6250 | F1 0.6964
Epoch   80 | Loss 0.7043/0.6003 | Sens 0.9535 | Spec 0.6125 | F1 0.7130
Epoch  100 | Loss 0.7015/0.5916 | Sens 0.9535 | Spec 0.6250 | F1 0.7193


#### Threshold Analysis

In [30]:
model.load_state_dict(torch.load('project2_diabetes/best_model.pth', map_location = DEVICE))
_, v_probs, v_lbls = run_epoch_diabetes(model, val_loader, criterion)

print(f'{'Thresh':>7} {'Sens':>8} {'Spec':>8} {'PPV':>8} {'NPV':>8} {'F1':>8}')

for i in np.arange(0.2, 0.85, 0.1):
    m = clinical_metrics(v_lbls, v_probs, threshold = i)
    print(f'{i:>7.2f} {m['sensitivity']:>8.4f} {m['specificity']:>8.4f} '
         f'{m['ppv']:>8.4f} {m['npv']:>8.4f} {m['f1']:>8.4f}')

 Thresh     Sens     Spec      PPV      NPV       F1
   0.20   0.9767   0.4375   0.4828   0.9722   0.6462
   0.30   0.9767   0.5250   0.5250   0.9767   0.6829
   0.40   0.9070   0.6875   0.6094   0.9322   0.7290
   0.50   0.8372   0.7500   0.6429   0.8955   0.7273
   0.60   0.7442   0.8625   0.7442   0.8625   0.7442
   0.70   0.5814   0.9500   0.8621   0.8085   0.6944
   0.80   0.3023   0.9750   0.8667   0.7222   0.4483


#### SHAP Interpretability

In [36]:
model.eval()

def model_predict(X_np):
    X = torch.tensor(X_np, dtype = torch.float32).to(DEVICE)
    with torch.no_grad():
        return torch.sigmoid(model(X)).cpu().numpy()

    bg_size = min(50, len(X_train_s))
    bg = X_train_s[: bg_size]

    explainer = shap.KernelExplainer(model_predict, bg)
    shap_values = explainer.shap_values(X_val_s[: 20], nsamples = 100)

    mean_abs_shap = np.abs(shap.values).mean(axis = 0)
    sorted_idx = np.argsort(mean_abs_shap)[:: -1]

    print(f'{'Feature':<30} {'Importance':>12} {'Bar':}')
    for i in sorted_idx:
        bar = '🟢' * int(mean_abs_shap[i] / mean_abs_shap.max() * 25)
        print(f'{FEATURE_NAMES[i]:<28} {mean_abs_shap[i]:>12.4f}  {bar}')

    sv = shap_values[0] if len(shap_values.shape) > 1 else shap_values
    base_val = explainer.expected_values
    pred_val = model_predict[X_val_s[:1]][0, 0]

    print(f'Base Pred (avg): {float(base_val):.4f}')
    print(f'Model Pred: {float(pred_val):.4f}')
    print(f'True Label: {y_val[0]}')

    print(f'{'Feature':<30} {'Value':>8} {'SHAP':>10}')
    for i in np.argsort(np.abs(sv))[:: -1]:
        direction = 'More Risk' if sv[i] > 0 else 'Lowered Risk'
        print(f'{FEATURE_NAMES[i]:<30} {X_val[0, i]:>8.2f} {sv[i]:>+10.4f} {direction}')

#### Calibration Check

In [40]:
print("A well-calibrated model: if it says 70% probability, 70% of those patients are diabetic.")
_, test_probs, test_lbls = run_epoch_diabetes(model, test_loader, criterion)

n_bins = 5
bins = np.linspace(0, 1, n_bins + 1)
print(f"{'Prob Range':>14} {'Pred Prob':>11} {'Actual Rate':>13} {'Count':>7} {'Calibrated?':>13}")

for i in range(n_bins):
    lo, hi = bins[i], bins[i + 1]
    mask = (test_probs >= lo) & (test_probs < hi)

    if mask.sum() == 0:
        continue

    mean_pred = test_probs[mask].mean()
    actual_rate = test_lbls[mask].mean()
    count = mask.sum()
    diff = abs(mean_pred - actual_rate)
    calibrated  = "✅" if diff < 0.1 else "💀 off"
    print(f"  [{lo:.1f} – {hi:.1f}]:  "
          f"{mean_pred:>9.4f}  {actual_rate:>11.4f}  {count:>7}  {calibrated:>13}")

print("If poorly calibrated: consider temperature scaling or isotonic regression.")

A well-calibrated model: if it says 70% probability, 70% of those patients are diabetic.
    Prob Range   Pred Prob   Actual Rate   Count   Calibrated?
  [0.0 – 0.2]:     0.1030       0.0263       38              ✅
  [0.2 – 0.4]:     0.2816       0.2121       33              ✅
  [0.4 – 0.6]:     0.5011       0.4848       33              ✅
  [0.6 – 0.8]:     0.7083       0.5172       29          💀 off
  [0.8 – 1.0]:     0.8767       0.7143       21          💀 off
If poorly calibrated: consider temperature scaling or isotonic regression.


#### Final Test Metrics

In [41]:
test_m = clinical_metrics(test_lbls, test_probs, threshold=0.35)
print(f"{'Metric':<20} {'Value':>10}  Clinical Meaning")

print(f"  {'Sensitivity':<18} {test_m['sensitivity']:>10.4f}  Of all diabetics, this % caught")
print(f"  {'Specificity':<18} {test_m['specificity']:>10.4f}  Of non-diabetics, this % correctly cleared")
print(f"  {'PPV':<18} {test_m['ppv']:>10.4f}  When flagged, probability truly diabetic")
print(f"  {'NPV':<18} {test_m['npv']:>10.4f}  When cleared, probability truly healthy")
print(f"  {'Accuracy':<18} {test_m['accuracy']:>10.4f}")
print(f"  {'F1':<18} {test_m['f1']:>10.4f}")

lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_train_s, y_train)
lr_probs = lr_baseline.predict_proba(X_test_s)[:, 1]
lr_m     = clinical_metrics(y_test, lr_probs, threshold=0.35)

print(f"\n{'':23} {'ANN':>10} {'LogReg':>10}")
print("─" * 45)
for i in ['sensitivity', 'specificity', 'ppv', 'f1']:
    print(f"  {i:<21} {test_m[i]:>10.4f} {lr_m[i]:>10.4f}")

Metric                    Value  Clinical Meaning
  Sensitivity            0.8704  Of all diabetics, this % caught
  Specificity            0.5900  Of non-diabetics, this % correctly cleared
  PPV                    0.5341  When flagged, probability truly diabetic
  NPV                    0.8939  When cleared, probability truly healthy
  Accuracy               0.6883
  F1                     0.6620

                               ANN     LogReg
─────────────────────────────────────────────
  sensitivity               0.8704     0.7037
  specificity               0.5900     0.7500
  ppv                       0.5341     0.6032
  f1                        0.6620     0.6496
